# Erlang C Security Tests

This notebook performs safe, local security checks through FastAPI TestClient. No external server is required and it does not change your real datasets.

It focuses on validation and safe error handling. Authentication is intentionally out of scope because the current application is designed for local use.

## Checks included

1. Unsupported file type rejection
2. Empty file rejection
3. File larger than 25 MB rejection
4. More than 10 files rejection
5. Invalid forecast-days rejection
6. Invalid maximum-agent count rejection
7. Invalid month rejection
8. Incomplete schedule rejection
9. Incomplete leave request rejection
10. Empty swap schedule rejection
11. Path-like uploaded filename is not used as a server path
12. Validation errors do not reveal Python tracebacks

In [1]:
import sys
from io import BytesIO
from pathlib import Path
from datetime import datetime
import pandas as pd
from fastapi.testclient import TestClient
from IPython.display import Markdown, display

cwd = Path.cwd()
PROJECT_DIR = cwd if (cwd / 'api.py').exists() else cwd.parent
if not (PROJECT_DIR / 'api.py').exists():
    raise FileNotFoundError('Could not find api.py. Place this notebook inside the tests folder.')
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from api import app
client = TestClient(app)
SECURITY_RESULTS = []

def record(test_id, description, passed, expected, actual, details=''):
    SECURITY_RESULTS[:] = [row for row in SECURITY_RESULTS if row['test'] != test_id]
    status = 'PASS' if passed else 'FAIL'
    SECURITY_RESULTS.append({'test': test_id, 'description': description, 'status': status, 'expected': str(expected), 'actual': str(actual), 'details': details})
    print(f'{status}: {test_id} — {description}')
    print('Expected:', expected)
    print('Actual:  ', actual)
    if details: print('Details: ', details)

def post_forecast(files, **fields):
    defaults = {
        'interval_minutes': '30', 'forecast_days': '1', 'trend_lookback_days': '7',
        'target_seconds': '20', 'target_service_level': '80', 'shrinkage': '30',
        'max_agents': '100', 'include_forecast_rows': 'false',
    }
    defaults.update({key: str(value) for key, value in fields.items()})
    return client.post('/api/v1/cdr/stl-forecast', files=files, data=defaults)

minimal_forecast = [{'interval_start': '2025-01-10T08:00:00', 'scheduled_agents': 1}]
minimal_schedule = [{'agent_id': 'Agent 001', 'date': '2025-01-10', 'shift_code': 'MORNING', 'shift': 'Morning (08:00-16:00)', 'status': 'WORK'}]
print('Setup complete')
print('Python:', sys.executable)
print('Project directory:', PROJECT_DIR)

c:\Users\adept\Desktop\ADEPT\ERLANG\Calculator-Erlang-C\.venv\Lib\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


Setup complete
Python: c:\Users\adept\Desktop\ADEPT\ERLANG\Calculator-Erlang-C\.venv\Scripts\python.exe
Project directory: c:\Users\adept\Desktop\ADEPT\ERLANG\Calculator-Erlang-C


## SEC-01 to SEC-04 — Upload protection

In [2]:
response = post_forecast([('files', ('unsafe.xlsx', BytesIO(b'test'), 'application/vnd.openxmlformats-officedocument.spreadsheetml.sheet'))])
record('SEC-01', 'Excel upload is rejected', response.status_code == 400, 'HTTP 400', response.status_code, response.text)

response = post_forecast([('files', ('empty.csv', BytesIO(b''), 'text/csv'))])
record('SEC-02', 'Empty CSV upload is rejected', response.status_code == 400, 'HTTP 400', response.status_code, response.text)

oversized = BytesIO(b'x' * (25 * 1024 * 1024 + 1))
response = post_forecast([('files', ('oversized.csv', oversized, 'text/csv'))])
record('SEC-03', 'File larger than 25 MB is rejected', response.status_code == 413, 'HTTP 413', response.status_code, response.text)

many_files = [('files', (f'file_{number}.csv', BytesIO(b'a,b\n1,2\n'), 'text/csv')) for number in range(11)]
response = post_forecast(many_files)
record('SEC-04', 'Eleven uploaded files are rejected', response.status_code == 400, 'HTTP 400', response.status_code, response.text)

PASS: SEC-01 — Excel upload is rejected
Expected: HTTP 400
Actual:   400
Details:  {"detail":"Unsupported file type for unsafe.xlsx."}
PASS: SEC-02 — Empty CSV upload is rejected
Expected: HTTP 400
Actual:   400
Details:  {"detail":"empty.csv is empty."}
PASS: SEC-03 — File larger than 25 MB is rejected
Expected: HTTP 413
Actual:   413
Details:  {"detail":"Uploaded files must be 25 MB or smaller."}
PASS: SEC-04 — Eleven uploaded files are rejected
Expected: HTTP 400
Actual:   400
Details:  {"detail":"Upload no more than 10 files at a time."}


## SEC-05 to SEC-07 — Numeric request validation

In [3]:
sample = [('files', ('valid.csv', BytesIO(b'call_id,date,duration,disposition\n1,2024-Jan-01 01:00:00 AM,00:00:01,ANSWERED\n'), 'text/csv'))]
response = post_forecast(sample, forecast_days=0)
record('SEC-05', 'forecast_days=0 is rejected', response.status_code == 400, 'HTTP 400', response.status_code, response.text)

sample = [('files', ('valid.csv', BytesIO(b'call_id,date,duration,disposition\n1,2024-Jan-01 01:00:00 AM,00:00:01,ANSWERED\n'), 'text/csv'))]
response = post_forecast(sample, forecast_days=3651)
record('SEC-06', 'forecast_days=3651 is rejected', response.status_code == 400, 'HTTP 400', response.status_code, response.text)

sample = [('files', ('valid.csv', BytesIO(b'call_id,date,duration,disposition\n1,2024-Jan-01 01:00:00 AM,00:00:01,ANSWERED\n'), 'text/csv'))]
response = post_forecast(sample, max_agents=10001)
record('SEC-07', 'max_agents=10001 is rejected', response.status_code == 400, 'HTTP 400', response.status_code, response.text)

PASS: SEC-05 — forecast_days=0 is rejected
Expected: HTTP 400
Actual:   400
Details:  {"detail":"forecast_days must be between 1 and 3650."}
PASS: SEC-06 — forecast_days=3651 is rejected
Expected: HTTP 400
Actual:   400
Details:  {"detail":"forecast_days must be between 1 and 3650."}
PASS: SEC-07 — max_agents=10001 is rejected
Expected: HTTP 400
Actual:   400
Details:  {"detail":"max_agents must be between 1 and 10000."}


## SEC-08 to SEC-10 — JSON payload validation

In [4]:
response = client.post('/api/v1/schedule/monthly', json={'forecast': minimal_forecast, 'year': 2025, 'month': 13})
record('SEC-08', 'Month 13 is rejected by request validation', response.status_code == 422, 'HTTP 422', response.status_code, response.text)

response = client.post('/api/v1/schedule/leave', json={'forecast': minimal_forecast, 'schedule': [{'agent_id': 'Agent 001'}], 'agent_id': 'Agent 001', 'leave_date': '2025-01-10'})
record('SEC-09', 'Incomplete leave schedule is rejected', response.status_code == 400, 'HTTP 400', response.status_code, response.text)

response = client.post('/api/v1/schedule/swap', json={'schedule': [], 'agent_1': 'Agent 001', 'agent_2': 'Agent 002', 'swap_date': '2025-01-10'})
record('SEC-10', 'Empty swap schedule is rejected', response.status_code == 422, 'HTTP 422', response.status_code, response.text)

PASS: SEC-08 — Month 13 is rejected by request validation
Expected: HTTP 422
Actual:   422
Details:  {"detail":[{"type":"less_than_equal","loc":["body","month"],"msg":"Input should be less than or equal to 12","input":13,"ctx":{"le":12}}]}
PASS: SEC-09 — Incomplete leave schedule is rejected
Expected: HTTP 400
Actual:   400
Details:  {"detail":"Schedule is missing required columns: ['date', 'shift', 'shift_code', 'status']"}
PASS: SEC-10 — Empty swap schedule is rejected
Expected: HTTP 422
Actual:   422
Details:  {"detail":[{"type":"too_short","loc":["body","schedule"],"msg":"List should have at least 1 item after validation, not 0","input":[],"ctx":{"field_type":"List","min_length":1,"actual_length":0}}]}


## SEC-11 and SEC-12 — Filename handling and safe errors

In [5]:
path_like_name = '../../not_a_server_path.xlsx'
response = post_forecast([('files', (path_like_name, BytesIO(b'test'), 'application/vnd.openxmlformats-officedocument.spreadsheetml.sheet'))])
safe_filename_handling = response.status_code == 400 and 'not_a_server_path' in response.text and 'Traceback' not in response.text
record('SEC-11', 'Path-like filename is handled safely', safe_filename_handling, 'HTTP 400 without traceback', response.status_code, response.text)

response = client.post('/api/v1/schedule/monthly', json={'forecast': [], 'year': 2025, 'month': 1})
safe_error = response.status_code in {400, 422} and 'Traceback' not in response.text and 'File ' not in response.text
record('SEC-12', 'Invalid request does not expose a Python traceback', safe_error, 'Safe 4xx response without traceback', response.status_code, response.text)

PASS: SEC-11 — Path-like filename is handled safely
Expected: HTTP 400 without traceback
Actual:   400
Details:  {"detail":"Unsupported file type for ../../not_a_server_path.xlsx."}
PASS: SEC-12 — Invalid request does not expose a Python traceback
Expected: Safe 4xx response without traceback
Actual:   422
Details:  {"detail":[{"type":"too_short","loc":["body","forecast"],"msg":"List should have at least 1 item after validation, not 0","input":[],"ctx":{"field_type":"List","min_length":1,"actual_length":0}}]}


## Final security report

In [6]:
expected_ids = [f'SEC-{number:02d}' for number in range(1, 13)]
completed = {row['test'] for row in SECURITY_RESULTS}
passed = sum(row['status'] == 'PASS' for row in SECURITY_RESULTS)
failed = sum(row['status'] == 'FAIL' for row in SECURITY_RESULTS)
not_run = [test_id for test_id in expected_ids if test_id not in completed]
rows = []
for test_id in expected_ids:
    row = next((item for item in SECURITY_RESULTS if item['test'] == test_id), None)
    if row:
        rows.append(f"| {row['test']} | {row['description']} | {row['expected']} | {row['actual']} | {row['status']} |")
    else:
        rows.append(f'| {test_id} | Not run | — | — | NOT RUN |')
final_status = 'PASS' if passed == len(expected_ids) else ('FAIL' if failed else 'INCOMPLETE')
report = f'''
# Erlang C Security Testing Report

## Test information

- Test date: {datetime.now().strftime('%Y-%m-%d %H:%M')}
- Test method: FastAPI TestClient
- External server required: No
- Scope: Local-input validation and safe error handling
- Authentication: Not implemented because the application is intended for local use

## Overall results

| Result | Count |
|---|---:|
| Expected tests | {len(expected_ids)} |
| Passed | {passed} |
| Failed | {failed} |
| Not run | {len(not_run)} |

## Detailed results

| Test | Description | Expected | Actual | Result |
|---|---|---|---|---|
{chr(10).join(rows)}

## Scope note

This is a basic local-application security test. It does not prove production security. If the system is deployed publicly, add authentication, authorization, HTTPS, rate limiting, logging, and dependency scanning.

## Final status

{final_status}

{'All selected local security checks passed.' if final_status == 'PASS' else ('One or more security checks failed and should be reviewed.' if final_status == 'FAIL' else 'Run every cell before finalizing.')}
'''
display(Markdown(report))
report_path = PROJECT_DIR / 'tests' / 'Erlang_C_Security_Test_Report.md'
report_path.write_text(report, encoding='utf-8')
print('Report saved to:', report_path)


# Erlang C Security Testing Report

## Test information

- Test date: 2026-09-05 11:18
- Test method: FastAPI TestClient
- External server required: No
- Scope: Local-input validation and safe error handling
- Authentication: Not implemented because the application is intended for local use

## Overall results

| Result | Count |
|---|---:|
| Expected tests | 12 |
| Passed | 12 |
| Failed | 0 |
| Not run | 0 |

## Detailed results

| Test | Description | Expected | Actual | Result |
|---|---|---|---|---|
| SEC-01 | Excel upload is rejected | HTTP 400 | 400 | PASS |
| SEC-02 | Empty CSV upload is rejected | HTTP 400 | 400 | PASS |
| SEC-03 | File larger than 25 MB is rejected | HTTP 413 | 413 | PASS |
| SEC-04 | Eleven uploaded files are rejected | HTTP 400 | 400 | PASS |
| SEC-05 | forecast_days=0 is rejected | HTTP 400 | 400 | PASS |
| SEC-06 | forecast_days=3651 is rejected | HTTP 400 | 400 | PASS |
| SEC-07 | max_agents=10001 is rejected | HTTP 400 | 400 | PASS |
| SEC-08 | Month 13 is rejected by request validation | HTTP 422 | 422 | PASS |
| SEC-09 | Incomplete leave schedule is rejected | HTTP 400 | 400 | PASS |
| SEC-10 | Empty swap schedule is rejected | HTTP 422 | 422 | PASS |
| SEC-11 | Path-like filename is handled safely | HTTP 400 without traceback | 400 | PASS |
| SEC-12 | Invalid request does not expose a Python traceback | Safe 4xx response without traceback | 422 | PASS |

## Scope note

This is a basic local-application security test. It does not prove production security. If the system is deployed publicly, add authentication, authorization, HTTPS, rate limiting, logging, and dependency scanning.

## Final status

PASS

All selected local security checks passed.


Report saved to: c:\Users\adept\Desktop\ADEPT\ERLANG\Calculator-Erlang-C\tests\Erlang_C_Security_Test_Report.md
